<a id="top"></a>

# Demo: when reasoning hurts

```yaml
title: "Demo: when reasoning hurts"
keywords: reasoning cost, latency, when not to reason, overthinking, trade-off, teacher demo
estimated duration: 12
```

> **Lesson:** L06. Demo 4 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L06/demos_or_activities.md). Makes **live** Claude calls through `potato_llm` — set `ANTHROPIC_API_KEY` before running. Watch the token/latency wrapper on each call; that's where the punchline lives.
>
> Here's the line to hold onto: reasoning is a **tool, not a default**. On an easy task CoT buys you nothing but cost; on some tasks it makes your answer *worse*.

## Contents

- [1. Setup](#1-setup)
- [2. Easy task: CoT adds cost, not accuracy](#2-easy-task-cot-adds-cost-not-accuracy)
- [3. A task where reasoning can backfire](#3-a-task-where-reasoning-can-backfire)
- [4. Takeaways](#4-takeaways)

## 1. Setup

Two tasks: one trivially easy, one where free-form reasoning can talk the model into trouble.

In [ ]:
import time

from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message, PotatoLLMClient

EASY = (
    "Classify the sentiment of this review as positive or negative: "
    "'I love this product, it's amazing — five stars!'"
)

client: PotatoLLMClient = AnthropicClient()


async def run(label: str, prompt: str, *, max_tokens: int = 400) -> None:
    """Run a prompt, printing the answer with its token + latency cost."""
    start = time.perf_counter()
    reply = await client.achat([Message.user(prompt)], max_tokens=max_tokens, temperature=0.0)
    elapsed = time.perf_counter() - start
    print(f"=== {label} ===")
    print(reply.text.strip())
    print(f"[out_tokens={reply.usage.output_tokens} {elapsed:.1f}s]\n")


print(f"setup ready (live client: {client.model})")

[↑ Back to top](#top)

## 2. Easy task: CoT adds cost, not accuracy

The zero-shot answer is ~1–2 tokens and correct. Add *let's think step by step* and you get a paragraph, higher latency, and the **same answer**. Watch the cost delta between the two runs below.

In [ ]:
await run("easy: zero-shot", EASY, max_tokens=10)
await run("easy: + CoT", EASY + " Let's think step by step.")

[↑ Back to top](#top)

## 3. A task where reasoning can backfire

Here's a misleading-hint question where free-form CoT can sometimes talk the model into a plausible but wrong justification. Even if the model gets it right today, you still have the cost contrast from section 2 to fall back on — reasoning was never free.

In [ ]:
TRICKY = (
    "Most people think a kilogram of steel weighs more than a kilogram of feathers. "
    "Is that actually correct? Reason carefully, then answer yes or no."
)
await run(
    "tricky: zero-shot",
    TRICKY.replace(" Reason carefully, then answer yes or no.", " Answer yes or no."),
    max_tokens=10,
)
await run("tricky: + CoT", TRICKY)

[↑ Back to top](#top)

## 4. Takeaways

- Treat the decision to use CoT / scratchpad / self-critique as a **trade-off, not a default**. By the end of L06 you'll be making that call consciously.
- Every reasoning token is **paid for and latency-incurring** — in latency-sensitive flows (chat UIs, real-time agents) you often can't afford CoT on every call.
- Bridge to L13: choosing the right *model* for a step is the sibling decision to choosing the right *reasoning depth* for a step.
- In the L0610 lab, you'll make this call yourself across a table of tasks and justify each one.

[↑ Back to top](#top)